In [3]:
#imports 
from pyspark.sql import SparkSession
from pyspark.sql.types import * 

from pyspark.sql.functions import * 


In [18]:

# Start a spark session 
try : 
    spark.stop()
except : 
    pass 
spark = SparkSession\
        .builder\
        .master('spark://172.22.0.2:7077')\
        .appName('Process Sanfrancesco Fire Departement Data')\
        .getOrCreate()

print(f'spark version : {spark.version}')
print(f'application name is : {spark.sparkContext.appName}')


spark version : 3.5.1
application name is : Process Sanfrancesco Fire Departement Data


#### 1. Read/Write Data :

In [19]:
# Set up the data source file's path 
sf_fire_file = '/opt/spark-data/sf-fire-calls.csv'

In [20]:
# Define the schema of the data set 

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
StructField('UnitID', StringType(), True),
StructField('IncidentNumber', IntegerType(), True),
StructField('CallType', StringType(), True),                  
StructField('CallDate', StringType(), True),      
StructField('WatchDate', StringType(), True),
StructField('CallFinalDisposition', StringType(), True),
StructField('AvailableDtTm', StringType(), True),
StructField('Address', StringType(), True),       
StructField('City', StringType(), True),       
StructField('Zipcode', IntegerType(), True),       
StructField('Battalion', StringType(), True),                 
StructField('StationArea', StringType(), True),       
StructField('Box', StringType(), True),       
StructField('OriginalPriority', StringType(), True),       
StructField('Priority', StringType(), True),       
StructField('FinalPriority', IntegerType(), True),       
StructField('ALSUnit', BooleanType(), True),       
StructField('CallTypeGroup', StringType(), True),
StructField('NumAlarms', IntegerType(), True),
StructField('UnitType', StringType(), True),
StructField('UnitSequenceInCallDispatch', IntegerType(), True),
StructField('FirePreventionDistrict', StringType(), True),
StructField('SupervisorDistrict', StringType(), True),
StructField('Neighborhood', StringType(), True),
StructField('Location', StringType(), True),
StructField('RowID', StringType(), True),
StructField('Delay', FloatType(), True)])

In [21]:
# load dataframe with San Francesco fire data 

fire_df = spark.read.csv(sf_fire_file, header= True, schema = fire_schema)
        #.option('samplingRatio', 0.00001)\ # to infere schema

print(f'the number of columns is {len(fire_df.columns)}')
print(f'the number of rows is : {fire_df.count()}')

the number of columns is 28


[Stage 0:>                                                        (0 + 11) / 11]

the number of rows is : 175296


#### 2. Select and Filter Data : 

In [9]:
# df.select('Incident Number', 'Incident Date', 'Call Number' ).where(col('Incident Date').isNotNull()).show(10)

few_fire_df = fire_df\
    .select("IncidentNumber", "AvailableDtTm", "CallType") \
    .where(col("CallType") != "Medical Incident")

few_fire_df.show(5, truncate=False)

+--------------+----------------------+--------------+
|IncidentNumber|AvailableDtTm         |CallType      |
+--------------+----------------------+--------------+
|2003235       |01/11/2002 01:51:44 AM|Structure Fire|
|2003250       |01/11/2002 04:16:46 AM|Vehicle Fire  |
|2003259       |01/11/2002 06:01:58 AM|Alarms        |
|2003279       |01/11/2002 08:03:26 AM|Structure Fire|
|2003301       |01/11/2002 09:46:44 AM|Alarms        |
+--------------+----------------------+--------------+
only showing top 5 rows



In [10]:
fire_df.select('CallType').where(col('CallType').isNotNull()).agg(countDistinct('CallType').alias('DistinctCallTypes')).show()

[Stage 4:>                                                        (0 + 11) / 11]

+-----------------+
|DistinctCallTypes|
+-----------------+
|               30|
+-----------------+



In [11]:
fire_df.select('CallType').distinct().show(30, False)

+--------------------------------------------+
|CallType                                    |
+--------------------------------------------+
|Elevator / Escalator Rescue                 |
|Aircraft Emergency                          |
|Alarms                                      |
|Odor (Strange / Unknown)                    |
|Citizen Assist / Service Call               |
|HazMat                                      |
|Explosion                                   |
|Oil Spill                                   |
|Vehicle Fire                                |
|Suspicious Package                          |
|Extrication / Entrapped (Machinery, Vehicle)|
|Other                                       |
|Outside Fire                                |
|Traffic Collision                           |
|Assist Police                               |
|Gas Leak (Natural and LP Gases)             |
|Water Rescue                                |
|Electrical Hazard                           |
|High Angle R

#### 3. Aggregations : GroupBy, orderBy, and count 

- let's answere : what were the most common types of fire calls ?

In [12]:
fire_df.select('CallType')\
    .where(col('CallType').isNotNull())\
    .groupBy('CallType').count()\
    .orderBy('count', ascending = False)\
    .show(10, False)

+-------------------------------+------+
|CallType                       |count |
+-------------------------------+------+
|Medical Incident               |113794|
|Structure Fire                 |23319 |
|Alarms                         |19406 |
|Traffic Collision              |7013  |
|Citizen Assist / Service Call  |2524  |
|Other                          |2166  |
|Outside Fire                   |2094  |
|Vehicle Fire                   |854   |
|Gas Leak (Natural and LP Gases)|764   |
|Water Rescue                   |755   |
+-------------------------------+------+
only showing top 10 rows



##### count vs collect vs take : 

In [13]:
fire_df.select('CallType')\
    .where(col('CallType').isNotNull())\
    .groupBy('CallType').count()\
    .orderBy(col('count').desc())\
    .show(10, False)

+-------------------------------+------+
|CallType                       |count |
+-------------------------------+------+
|Medical Incident               |113794|
|Structure Fire                 |23319 |
|Alarms                         |19406 |
|Traffic Collision              |7013  |
|Citizen Assist / Service Call  |2524  |
|Other                          |2166  |
|Outside Fire                   |2094  |
|Vehicle Fire                   |854   |
|Gas Leak (Natural and LP Gases)|764   |
|Water Rescue                   |755   |
+-------------------------------+------+
only showing top 10 rows



In [14]:
# stop the session 

spark.stop()

In [15]:
## THIS IS THE END ##

In [16]:
display(fire_df)

DataFrame[CallNumber: int, UnitID: string, IncidentNumber: int, CallType: string, CallDate: string, WatchDate: string, CallFinalDisposition: string, AvailableDtTm: string, Address: string, City: string, Zipcode: int, Battalion: string, StationArea: string, Box: string, OriginalPriority: string, Priority: string, FinalPriority: int, ALSUnit: boolean, CallTypeGroup: string, NumAlarms: int, UnitType: string, UnitSequenceInCallDispatch: int, FirePreventionDistrict: string, SupervisorDistrict: string, Neighborhood: string, Location: string, RowID: string, Delay: float]